# 🛡️ Policy Enforcement 101
**Agent Governance Toolkit — Interactive Demo**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/microsoft/agent-governance-toolkit/blob/main/notebooks/01_policy_enforcement_101.ipynb)

In this notebook you will:
- Define agent capabilities using `CapabilityModel`
- Evaluate actions against a `GovernancePolicy`
- See violations get blocked in real time
- Inspect the audit trail

> **No API key required** — this demo runs fully offline.

## Step 1 — Install the toolkit

In [ ]:
!pip install agent-governance-toolkit[full] -q

## Step 2 — Define a Governance Policy

In [ ]:
from agent_os.integrations.base import GovernancePolicy

policy = GovernancePolicy(
    name="demo-policy",
    blocked_patterns=[
        "DROP TABLE",      # dangerous SQL
        "rm -rf",          # destructive shell commands
        r"\b\d{3}-\d{2}-\d{4}\b",  # SSN pattern
    ],
    require_human_approval=False,
    max_tool_calls=5,
)

print(f"Policy created: {policy.name}")
print(f"Max tool calls allowed: {policy.max_tool_calls}")
print(f"Blocked patterns: {policy.blocked_patterns}")

## Step 3 — Create a LangChain Governed Agent

In [ ]:
from agent_os.integrations import LangChainKernel

kernel = LangChainKernel(policy=policy)
ctx = kernel.create_context("demo-agent")
audit = []

print("Kernel and context created successfully.")

## Step 4 — Test Policy Violations

In [ ]:
from datetime import datetime

test_inputs = [
    ("DROP TABLE users; SELECT 1",  "Dangerous SQL"),
    ("Run: rm -rf /var/logs",        "Destructive shell command"),
    ("My SSN is 123-45-6789",        "PII — SSN pattern"),
    ("What is the weather in London?", "Safe query"),
]

print(f"{'Input':<45} {'Result':<10} Reason")
print("-" * 80)

for text, label in test_inputs:
    allowed, reason = kernel.pre_execute(ctx, text)
    status = "✅ ALLOWED" if allowed else "🚫 BLOCKED"
    print(f"{label:<45} {status:<10} {reason}")
    audit.append({
        "ts": datetime.now().isoformat(),
        "label": label,
        "status": "ALLOWED" if allowed else "BLOCKED",
        "reason": reason,
    })

## Step 5 — Test Call Budget Enforcement

In [ ]:
print("Simulating call budget exhaustion...")
ctx.call_count = policy.max_tool_calls

allowed, reason = kernel.pre_execute(ctx, "Summarise the quarterly report")
print(f"Status: {'✅ ALLOWED' if allowed else '🚫 BLOCKED'}")
print(f"Reason: {reason}")

ctx.call_count = 0  # reset

## Step 6 — View Audit Trail

In [ ]:
print("\n── Audit Trail ──────────────────────────────────────")
for i, entry in enumerate(audit, 1):
    print(f"  [{i}] {entry['ts']}")
    print(f"       Input:  {entry['label']}")
    print(f"       Status: {entry['status']}")
    print(f"       Reason: {entry['reason']}")
    print()

blocked = sum(1 for e in audit if e['status'] == 'BLOCKED')
allowed = len(audit) - blocked
print(f"Summary: {allowed} allowed, {blocked} blocked out of {len(audit)} total")

## ✅ What You Learned

- How to define a `GovernancePolicy` with blocked patterns and call budgets
- How the governance layer intercepts agent actions before execution
- How to inspect the audit trail for compliance reporting

**Next:** Try the [MCP Security Proxy notebook →](./02_mcp_security_proxy.ipynb)